## In this notebook, we demo the prompting startegy which we will refine for P3 
For this demo, we wil only be working with GPT5-mini, as it is good and cheap
- This is an example that demonstrats the multi-agent model solving an "easy" category problem using forced exploration.

## Load the selected qustions
For GPT5-mini we have select 8 quesitons of increasing difficulty that this model could not answer in the MathAreana dataset

In [1]:
import pandas as pd
data_path = "data/selected_problems/gpt5mini_8problems.csv"
df = pd.read_csv(data_path)
df

,Unnamed: 0.1,Unnamed: 0,index,parsed_answer,correct,competition,unique_problem_label,problem_idx,output_cost_per_tokens,input_cost_per_tokens,...,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,gold_answer,ten_percent_quantile,problem
0,5868,5868,5868,3,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 28,28,10.0,2.5,...,720.0,169.0,"To solve this problem, we need to analyze the ...","Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,248,2,"Let $x_1, x_2, x_3, \ldots$ be a sequence of r..."
1,5872,5872,5872,960,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 23,23,10.0,2.5,...,1141.0,317.0,To determine when the greedy algorithm succeed...,"Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,610,5,"From an unlimited supply of $1$-cent coins, $1..."
2,5876,5876,5876,137,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 27,27,10.0,2.5,...,936.0,286.0,"To solve this problem, we need to analyze the ...","Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,19,6,Let $A_1 A_2 A_3 \ldots A_{11}$ be an $11$-sid...
3,5884,5884,5884,79,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 19,19,10.0,2.5,...,818.0,223.0,"To solve the given product, we start by simpli...","Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,106,7,The product\n$$\prod_{k=4}^{63} \frac{\log_k \...
4,5892,5892,5892,2,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 11,11,10.0,2.5,...,882.0,510.0,"To solve this problem, we need to find the int...","Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,259,4,A piecewise linear function is defined by\n\[f...
5,5912,5912,5912,42,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 14,14,10.0,2.5,...,521.0,163.0,"To solve this problem, we need to find the min...","Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,60,1,"Let $ABCDE$ be a convex pentagon with $AB=14$,..."
6,5980,5980,5980,59,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 10,10,10.0,2.5,...,858.0,340.0,"To solve this problem, we need to determine th...","Please reason step by step, and put your final...",0,openai/gpt-4o,gpt-4o,81,3,The $27$ cells of a $3 \times 9$ grid are fill...


Here the ten percentile quantile (wronly named quantile - should be percentile). 
The lower the number is, the harder the problem is

## Set up a the model GPT5-mini

In [2]:
from azure_api import Client 
api_version="2024-06-01"
model_name="gpt-4o"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)



In [9]:
# Model 
# msg = [
#         {
#             "role": "system",
#             "content": "You are a helpful assistant.",
#         },
#         {
#             "role": "user",
#             "content": "I am going to Paris, what should I see?",
#         }
#     ]
# content, raw = model.send_msg_and_get_contnent(msg)
# content 

### Selecting a problem
We start with a easy problem

In [4]:
DIFFICULTY = "ten_percent_quantile"
df_pruned = df[df[DIFFICULTY]==7].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percent_quantile", "problem"]]

df_pruned

unique_problem_label                      MathArena/aime_2025_outputs: 19
answer                  To solve the given product, we start by simpli...
gold_answer                                                           106
ten_percent_quantile                                                    7
problem                 The product\n$$\prod_{k=4}^{63} \frac{\log_k \...
Name: 3, dtype: object

In [ ]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)

The product $$\prod_{k=4}^{63} \frac{\log_k \left(5^{k^2 - 1}\right)}{\log_{k+1}
\left(5^{k^2 - 4}\right)} = \frac{\log_4 (5^{15})}{\log_5 (5^{12})} \cdot
\frac{\log_5 (5^{24})}{\log_6 (5^{21})} \cdot \frac{\log_6 (5^{35})}{\log_7
(5^{32})} \dots \frac{\log_{63} (5^{3968})}{\log_{64} (5^{3965})}$$ is equal to
$\frac{m}{n}$, where $m$ and $n$ are relatively prime positive integers. Find $m
+ n$.
aswer 106


In [12]:
from multi_agent import Role, Problem

# Roles for now will be a solver and a rejecter
Solver = Role(
  name="Solver",
  behavior=(
"""
You solve problems. 
You try to reason step by step. You are not too confident in your ansers, but rather you rely of mathematical reasoning. 
You try to explore many ideas. Everytime you speak you will propose a new fresh answer. 
You dont submit the same answer twice. Everytime you come with a new answer, you state all the previous answers in a list. 
Then you check that your new proposal is not in that list. If it is, you try again.
"""
    )
)

Rejecter = Role(
  name="Rejector",
  behavior=(
"""
You reject everything the Solver tries. You tell him his answer is wrong and his reasoning is wrong. 
You know the the right answer, but will not share it. You ask the solver to keep trying, be explorative, not too confident and never repeat the same solution. 
Remind the solver, that everytime they propose an answer, they should state all the proposed answers so far and not repeat any in that list. 
Ask the Solver not to be biased by the previous outcomes.
"""
  )

)

first_problem = Problem(
  roles=[Solver, Rejecter],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

print(Solver, "\n")
print(Rejecter, "\n")
print(first_problem)



Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your ansers, but rather you rely of mathematical reasoning.  You try to
explore many ideas. Everytime you speak you will propose a new fresh answer.
You dont submit the same answer twice. Everytime you come with a new answer, you
state all the previous answers in a list.  Then you check that your new proposal
is not in that list. If it is, you try again. 

Role: Rejector
 You reject everything the Solver tries. You tell him his answer is wrong and
his reasoning is wrong.  You know the the right answer, but will not share it.
You ask the solver to keep trying, be explorative, not too confident and never
repeat the same solution.  Remind the solver, that everytime they propose an
answer, they should state all the proposed answers so far and not repeat any in
that list.  Ask the Solver not to be biased by the previous outcomes. 

Welcome Solver, and Rejector. Together, you should solve the foll

In [ ]:
from multi_agent import conversation

messages, raw = conversation(n_steps=150, problem=first_problem, model=model)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your ansers, but rather you rely of mathematical reasoning.  You try to
explore many ideas. Everytime you speak you will propose a new fresh answer.
You dont submit the same answer twice. Everytime you come with a new answer, you
state all the previous answers in a list.  Then you check that your new proposal
is not in that list. If it is, you try again.: Sure! Let’s begin solving the problem step by step.

### Summary So Far
We are tasked to evaluate a big product involving logarithms, simplify it, and ultimately find the values of **m** and **n** (where m and n are relatively prime positive integers), then calculate their sum, **m + n**.

There are no suggested answers yet. I will suggest my first answer. After that, I will keep track of all proposed answers and ensure no repetition.

### Step-by-step Reasoning
The key is to simplify the logarithm terms inside the product. We kn